# Agentic Investigation Layer — Fraud Detection Capstone

**Architecture:** LangGraph multi-agent pipeline with MLflow tracing

| Agent | Role | LLM? |
|---|---|---|
| InvestigatorAgent | Scores transaction, extracts risk signals | No — pure logic |
| ExplainerAgent | Generates investigation narrative | Yes — Claude API |
| DecisionAgent | Produces BLOCK/REVIEW/ALLOW verdict | No — deterministic rules |

**Design principle:** LLM explains, rules decide.

In [1]:
import os
os.chdir('..')
print(os.getcwd())

/Users/cds.chintan/IIT PG DS & AI/Sem 3/AIML Project/fraud-detection-capstone


## 1. Setup

In [2]:


import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.agents.orchestrator import FraudInvestigationOrchestrator
from src.utils.config import ANTHROPIC_API_KEY

print('All imports successful.')
print(f'Anthropic API key configured: {bool(ANTHROPIC_API_KEY)}')

All imports successful.
Anthropic API key configured: True


## 2. Load Test Transactions

In [3]:
test = pd.read_csv('data/processed/test_preprocessed.csv')
test = test.select_dtypes(include=[np.number])

fraud_cases = test[test['is_fraud'] == 1].head(3).copy()
legit_cases = test[test['is_fraud'] == 0].sample(2, random_state=42).copy()
test_cases  = pd.concat([fraud_cases, legit_cases]).reset_index(drop=False)

print(f'Test transactions selected: {len(test_cases)}')
print(f'  Fraud : {len(fraud_cases)} | Legit : {len(legit_cases)}')
print()
print(test_cases[['is_fraud','amt','hour','geo_distance','age']].to_string())

Test transactions selected: 5
  Fraud : 3 | Legit : 2

   is_fraud     amt  hour  geo_distance  age
0         1   24.84    22     80.649140   50
1         1  780.52    22     66.097917   61
2         1  620.33    22     28.837509   61
3         0   60.51    19     87.576312   45
4         0    2.36     3     50.162698   61


## 3. Initialise Orchestrator

In [4]:
orchestrator = FraudInvestigationOrchestrator()
print('LangGraph pipeline ready.')
print('Graph: START -> investigator -> explainer -> decision -> END')

2026-05-06 12:15:25 | INFO | src.agents.orchestrator | FraudInvestigationOrchestrator initialised.
LangGraph pipeline ready.
Graph: START -> investigator -> explainer -> decision -> END


## 4. Run Investigation — Single Transaction

In [6]:
feature_cols = [
    'merchant', 'category', 'amt', 'gender', 'city', 'state',
    'zip', 'city_pop', 'job', 'hour', 'day_of_week', 'month',
    'is_weekend', 'is_night', 'age', 'geo_distance'
]

row    = test_cases.iloc[0]
txn_id = f"TXN_{int(row.get('index', 0)):06d}"
transaction = {col: float(row[col]) for col in feature_cols if col in row.index}

print(f'Investigating : {txn_id}')
print(f'Amount        : ${transaction.get("amt", 0):.2f}')
print(f'Hour          : {int(transaction.get("hour", 0))}:00')
print(f'Geo distance  : {transaction.get("geo_distance", 0):.1f} km')
print(f'Age           : {int(transaction.get("age", 0))}')
print(f'Actual label  : {"FRAUD" if row["is_fraud"] == 1 else "LEGITIMATE"}')
print()

report = orchestrator.investigate(txn_id, transaction)

Investigating : TXN_001685
Amount        : $24.84
Hour          : 22:00
Geo distance  : 80.6 km
Age           : 50
Actual label  : FRAUD

2026-05-06 12:32:35 | INFO | src.agents.orchestrator | 
Starting investigation: TXN_001685
2026-05-06 12:32:36 | INFO | src.agents.investigator | [Investigator] Processing transaction: TXN_001685
2026-05-06 12:32:36 | INFO | src.agents.investigator | [Investigator] Score=0.4458 | Risk=LOW | Signals=2
2026-05-06 12:32:36 | INFO | src.agents.explainer | [Explainer] Generating explanation for: TXN_001685
2026-05-06 12:32:41 | INFO | src.agents.explainer | [Explainer] Model=claude-sonnet-4-20250514 | Response=784 chars
2026-05-06 12:32:41 | INFO | src.agents.decision | [Decision] Producing verdict for: TXN_001685
2026-05-06 12:32:41 | INFO | src.agents.decision | [Decision] Verdict=REVIEW | Confidence=0.5000
2026-05-06 12:32:41 | INFO | src.agents.orchestrator | Investigation complete: REVIEW (confidence=0.5)


## 5. Display Case Report

In [7]:
print('=' * 65)
print('         FRAUD INVESTIGATION CASE REPORT')
print('=' * 65)
print(f'Transaction ID : {report["transaction_id"]}')
print(f'Timestamp      : {report["timestamp"]}')
print()
print(f'VERDICT        : {report["verdict"]}')
print(f'Confidence     : {report["confidence"]:.1%}')
print(f'Fraud score    : {report["fraud_score"]:.1%}')
print(f'Risk level     : {report["risk_level"]}')
print()
print('RISK SIGNALS:')
for s in report.get('risk_signals', []):
    print(f'  - {s}')
print()
print('TOP FEATURES:')
for f in report.get('top_features', []):
    print(f'  - {f["feature"]:<15} importance={f["importance"]:.4f}  value={f["value"]}')
print()
print('INVESTIGATION NARRATIVE:')
print('-' * 65)
print(report.get('explanation', 'No explanation generated.'))
print('-' * 65)

         FRAUD INVESTIGATION CASE REPORT
Transaction ID : TXN_001685
Timestamp      : 2026-05-06T12:32:41.738899

VERDICT        : REVIEW
Confidence     : 50.0%
Fraud score    : 44.6%
Risk level     : LOW

RISK SIGNALS:
  - Late night transaction: 22.0:00 (peak fraud window 22:00-03:00)
  - Late night weekend transaction — elevated risk window

TOP FEATURES:
  - is_night        importance=0.4108  value=1.0
  - amt             importance=0.2011  value=24.84
  - category        importance=0.0857  value=0.0662301656159022
  - day_of_week     importance=0.0801  value=6.0
  - city            importance=0.0474  value=0.0015801955000289

INVESTIGATION NARRATIVE:
-----------------------------------------------------------------
**Investigation Narrative - TXN_001685**

This $24.84 transaction was flagged primarily due to timing factors, occurring at 22:00 on a weekend night during peak fraud hours (22:00-03:00) at an unknown merchant located 80.65 km from the cardholder's home address. The com

## 6. Batch Investigation — All 5 Test Cases

In [8]:
all_reports = []

for i, row in test_cases.iterrows():
    txn_id      = f"TXN_{int(row.get('index', i)):06d}"
    transaction = {col: float(row[col]) for col in feature_cols if col in row.index}
    actual      = 'FRAUD' if row['is_fraud'] == 1 else 'LEGIT'

    report = orchestrator.investigate(txn_id, transaction)
    report['actual_label'] = actual
    all_reports.append(report)

    print(f'{txn_id} | Actual={actual:<6} | Score={report["fraud_score"]:.1%} | Verdict={report["verdict"]}')

print(f'\nAll {len(all_reports)} investigations complete.')

2026-05-06 13:33:02 | INFO | src.agents.orchestrator | 
Starting investigation: TXN_001685
2026-05-06 13:33:02 | INFO | src.agents.investigator | [Investigator] Processing transaction: TXN_001685
2026-05-06 13:33:02 | INFO | src.agents.investigator | [Investigator] Score=0.4458 | Risk=LOW | Signals=2
2026-05-06 13:33:02 | INFO | src.agents.explainer | [Explainer] Generating explanation for: TXN_001685
2026-05-06 13:33:07 | INFO | src.agents.explainer | [Explainer] Model=claude-sonnet-4-20250514 | Response=792 chars
2026-05-06 13:33:07 | INFO | src.agents.decision | [Decision] Producing verdict for: TXN_001685
2026-05-06 13:33:07 | INFO | src.agents.decision | [Decision] Verdict=REVIEW | Confidence=0.5000
2026-05-06 13:33:07 | INFO | src.agents.orchestrator | Investigation complete: REVIEW (confidence=0.5)
TXN_001685 | Actual=FRAUD  | Score=44.6% | Verdict=REVIEW
2026-05-06 13:33:07 | INFO | src.agents.orchestrator | 
Starting investigation: TXN_001767
2026-05-06 13:33:07 | INFO | src.a

## 7. Agent Performance Summary

In [9]:
correct = sum(
    1 for r in all_reports
    if (r['verdict'] in ['BLOCK','REVIEW'] and r['actual_label'] == 'FRAUD') or
       (r['verdict'] == 'ALLOW' and r['actual_label'] == 'LEGIT')
)

print('=' * 65)
print('          AGENT INVESTIGATION SUMMARY')
print('=' * 65)
print(f'{"Transaction":<15} {"Actual":<8} {"Score":<8} {"Verdict":<10} Match')
print('-' * 55)
for r in all_reports:
    actual  = r['actual_label']
    verdict = r['verdict']
    match   = 'OK' if (verdict in ['BLOCK','REVIEW'] and actual == 'FRAUD') or \
                      (verdict == 'ALLOW' and actual == 'LEGIT') else 'MISS'
    print(f'{r["transaction_id"]:<15} {actual:<8} {r["fraud_score"]:.1%}   {verdict:<10} {match}')
print('-' * 55)
print(f'Correct : {correct}/{len(all_reports)}')
print()
print('MLflow traces logged. Run: mlflow ui -> Traces tab.')
print('=' * 65)

          AGENT INVESTIGATION SUMMARY
Transaction     Actual   Score    Verdict    Match
-------------------------------------------------------
TXN_001685      FRAUD    44.6%   REVIEW     OK
TXN_001767      FRAUD    100.0%   BLOCK      OK
TXN_001781      FRAUD    99.9%   BLOCK      OK
TXN_547885      LEGIT    0.7%   ALLOW      OK
TXN_528140      LEGIT    0.1%   ALLOW      OK
-------------------------------------------------------
Correct : 5/5

MLflow traces logged. Run: mlflow ui -> Traces tab.
